# Notebook 04 — EDA: Occupancy Trends & Cross-Check (Day 15)

**Purpose:** Explore occupancy trends by segment independently of the SQL layer,
then cross-check that the pandas calculations match the SQL query results.

**Done-when:** Notebook runs end-to-end; pandas CoV numbers match SQL view output
for at least one spot-checked segment.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import sqlite3

REPO_ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import DB_PATH, FACT_TABLE_CSV, ASSUMED_TOTAL_ROOMS
print(f'DB path:   {DB_PATH}  (exists={DB_PATH.exists()})')
print(f'Fact CSV:  {FACT_TABLE_CSV}  (exists={FACT_TABLE_CSV.exists()})')

## 1. Load fact table from CSV

In [ ]:
fact = pd.read_csv(FACT_TABLE_CSV, parse_dates=['check_in_date','booking_date','check_out_date'])
print(f'Shape: {fact.shape}')
fact.head(3)

## 2. Daily occupancy rate by segment (pandas, Metric #1)

In [ ]:
active = fact[~fact['is_cancelled']].copy()
daily = (
    active.groupby(['segment', 'check_in_date'])['room_nights']
    .sum()
    .reset_index()
    .rename(columns={'room_nights': 'booked_room_nights'})
)
daily['occupancy_rate'] = daily['booked_room_nights'] / ASSUMED_TOTAL_ROOMS
print(daily.head(10).to_string(index=False))

## 3. Occupancy Volatility CoV per segment (pandas, Metric #2)

In [ ]:
cov_pandas = (
    daily.groupby('segment')['occupancy_rate']
    .agg(mean='mean', std='std')
    .assign(cov=lambda d: (d['std'] / d['mean']).round(4))
    .sort_values('cov', ascending=False)
)
print('CoV per segment (pandas):')
print(cov_pandas[['cov']].to_string())

## 4. Cross-check: pandas CoV vs SQL query result

In [ ]:
from src.queries import get_occupancy_volatility_cov
cov_sql = get_occupancy_volatility_cov(DB_PATH).set_index('segment_name')['cov']

print('CoV per segment (SQL queries.py):')
print(cov_sql.to_string())

print('\nDifference (pandas CoV - SQL CoV):')
diff = cov_pandas['cov'].subtract(cov_sql, fill_value=None)
print(diff.dropna().to_string())
print('\nMax absolute difference:', diff.dropna().abs().max())

## 5. Segment Volatility Contribution (Metric #6)

In [ ]:
var_by_seg = (
    daily.groupby('segment')['occupancy_rate']
    .var(ddof=1)
    .rename('variance')
)
total_var = var_by_seg.sum()
contribution = (var_by_seg / total_var).round(4).rename('volatility_contribution')
print('Volatility contribution per segment:')
print(contribution.sort_values(ascending=False).to_string())

## 6. Revenue at Risk by segment (Metric #7)

In [ ]:
cancelled = fact[fact['is_cancelled'] & fact['rate'].notna()].copy()
rar = (
    cancelled.assign(rev_at_risk=lambda d: d['room_nights'] * d['rate'])
    .groupby('segment')['rev_at_risk']
    .sum()
    .round(2)
    .sort_values(ascending=False)
)
print('Revenue at Risk (pandas):')
print(rar.to_string())

## 7. Cancellation rate by segment (Metric #3)

In [ ]:
cancel_rate = (
    fact.groupby('segment')['is_cancelled']
    .agg(total='count', cancelled='sum')
    .assign(rate=lambda d: (d['cancelled'] / d['total']).round(4))
    .sort_values('rate', ascending=False)
)
print('Cancellation rate (pandas):')
print(cancel_rate.to_string())

## 8. Summary findings

Run all cells above and fill in the table:

| Finding | Value |
|---------|-------|
| Most volatile segment (CoV) | |
| Highest volatility contribution | |
| Highest revenue at risk | |
| Highest cancellation rate | |
| Pandas vs SQL CoV max difference | |
